# 统计聚合与分布摘要

学习目标：按明确的轴、类型和缺失值口径生成统计摘要，核对均值、离散程度与分布位置，并处理空输入。

前置知识：数组形状与轴、广播、均值与方差的基本含义、布尔数组。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自制小数据，后续单元沿用首次导入的 np 和 warnings。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按列生成摘要

聚合把多个值合成一个结果。下面记录两天、三个站点的数量，形状为 (2, 3)。sum() 计算总量，mean() 计算算术平均值，即总和除以元素个数。

axis=0 表示沿天数轴汇总：每个站点留下一个结果，输出形状为 (3,)。

In [1]:
import warnings

import numpy as np

counts = np.array([[2, 4, 6], [4, 8, 12]], dtype=np.int64)
totals = np.sum(counts, axis=0)
means = np.mean(counts, axis=0)

print(totals)  # 三个站点分别为 [6 12 18]。
print(means)  # 分别除以两天，得到 [3. 6. 9.]。
print(totals.shape, totals.dtype)  # (3,) int64。
print(means.shape, means.dtype)  # (3,) float64，整数输入的 mean 默认用 float64。

[ 6 12 18]
[3. 6. 9.]
(3,) int64
(3,) float64


## 2 axis 与 keepdims

axis 指定被汇总的轴，通常从结果中移除它。沿天数轴 axis=0 求和，每个站点留下一个总量；沿站点轴 axis=1 求和，每天留下一个总量。axis=None 汇总全部元素，二维数组的 axis=-1 与 axis=1 相同。

![二天三站点的数组沿轴 0 汇总为三个站点总和，沿轴 1 汇总为每天总量；keepdims 保留长度一的被汇总轴。](image/illustration/10-01-reduction-axes.svg)

图中输出的横排或竖排用于对齐原位置，实际一维结果仍只有一个轴。keepdims=True 则明确把被汇总轴保留为长度 1。

下面继续使用 counts。先核对每天总量，再把每天均值保留成 (2, 1)，让减法按行复用这个均值；可沿图中的两行检查为什么不能误用 (2,) 来表达同一方向。

In [2]:
print(np.sum(counts, axis=1))  # 每天总量为 [12 24]，shape 为 (2,)。
print(np.sum(counts))  # 36，汇总全部六个值。

daily_means = np.mean(counts, axis=-1, keepdims=True)
deviations = counts - daily_means

print(daily_means, daily_means.shape)  # 两行一列，值为 4.0、8.0，shape 为 (2, 1)。
print(deviations)  # 两行为 [-2. 0. 2.]、[-4. 0. 4.]。
print(deviations.shape)  # (2, 3)，减法沿列重复使用每天的均值。

[12 24]
36
[[4.]
 [8.]] (2, 1)
[[-2.  0.  2.]
 [-4.  0.  4.]]
(2, 3)


## 3 聚合类型

sum() 和 prod() 的 dtype 同时决定累计计算与结果类型。较窄的整数输入默认会提升到平台整数类型，但显式指定过窄类型仍可能溢出，整数聚合不会因此自动报错。

mean() 对整数输入默认使用 float64 计算并返回结果。浮点输入通常沿用输入类型；float16 是需要区分的边界：默认用 float32 进行中间计算，再返回 float16 结果。结果的 dtype 不能单独说明中间计算精度。

可用 dtype 显式指定计算与结果类型。类型选择不能恢复输入在存储时已经丢失的信息。

In [3]:
counts = np.array([120, 120], dtype=np.int8)
narrow_total = np.sum(counts, dtype=np.int8)
wide_total = np.sum(counts, dtype=np.int64)

print(narrow_total, narrow_total.dtype)  # -16 int8，240 超出 int8 范围。
print(wide_total, wide_total.dtype)  # 240 int64。
print(np.sum(counts).dtype)  # 本机默认提升为 int64。

values = np.array([1.0, 2.0, 3.0], dtype=np.float32)
print(np.mean(values).dtype)  # float32。
print(np.mean(values, dtype=np.float64).dtype)  # float64，显式选择计算精度。

half_values = np.array([1.0, 2.0, 3.0], dtype=np.float16)
print(np.mean(half_values).dtype)  # float16；官方说明中间计算使用 float32。
print(np.mean(half_values, dtype=np.float32).dtype)  # float32，显式保留该结果类型。

-16 int8
240 int64
int64
float32
float64
float16
float32


## 4 乘积与累计计算

prod() 把所选元素相乘。cumsum() 和 cumprod() 分别保存截至每个位置的累计和、累计积，指定 axis 后保留输入形状。

累计函数默认 axis=None，会先按展平顺序累计，得到一维结果；二维输入应明确要沿哪个轴累计。

In [4]:
daily_counts = np.array([[2, 3, 4], [5, 6, 7]], dtype=np.int64)

print(np.prod(daily_counts, axis=1))  # 两行乘积为 [24 210]。
print(np.cumsum(daily_counts, axis=0))  # 首日 [2 3 4]，截至第二日 [7 9 11]。
print(np.cumsum(daily_counts, axis=1))  # 两行分别为 [2 5 9]、[5 11 18]。
print(np.cumprod(daily_counts, axis=1))  # 两行分别为 [2 6 24]、[5 30 210]。
print(np.cumsum(daily_counts).shape)  # (6,)，未指定轴时展平累计。

[ 24 210]
[[ 2  3  4]
 [ 7  9 11]]
[[ 2  5  9]
 [ 5 11 18]]
[[  2   6  24]
 [  5  30 210]]
(6,)


## 5 最值与位置

min()、max() 返回最小值与最大值；argmin()、argmax() 返回对应位置。指定 axis 时，位置是该轴内部的索引；未指定时返回展平后的位置。

最值多次出现时，位置函数返回第一次出现的位置。下面行表示两次观测，列表示三个通道。

In [5]:
readings = np.array([[4, 9, 9], [7, 2, 5]], dtype=np.int64)

print(np.min(readings, axis=1))  # 每行最小值：[4 2]。
print(np.max(readings, axis=1))  # 每行最大值：[9 7]。
print(np.argmin(readings, axis=1))  # 最小值所在列：[0 1]。
print(np.argmax(readings, axis=1))  # 最大值所在列：[1 0]，首行返回第一个 9 的位置。
print(np.argmax(readings))  # 1，展平后的第一个最大值位置。
print(np.max(readings, axis=0).shape)  # (3,)，每列保留一个最大值。

[4 2]
[9 7]
[0 1]
[1 0]
1
(3,)


## 6 加权平均

mean() 对每个元素等权。average() 可通过 weights 指定权重，计算“值与权重的乘积之和 ÷ 权重之和”。权重总和不能为 0。

本例的非负整数权重表示出现次数：10 出现一次，20 出现三次。对二维数组指定 axis=0 时，一维权重可以对应观测行。

In [6]:
values = np.array([10.0, 20.0])
weights = np.array([1.0, 3.0])

print(np.mean(values))  # 15.0，两项等权。
print(np.average(values, weights=weights))  # 17.5，(10×1 + 20×3) / 4。

observations = np.array([[10.0, 100.0], [20.0, 200.0]])
weighted = np.average(observations, axis=0, weights=weights)
print(weighted, weighted.shape)  # [17.5 175.] (2,)，每列使用相同的行权重。

15.0
17.5
[ 17.5 175. ] (2,)


## 7 方差、标准差与 ddof

var() 描述数值围绕均值的离散程度。对本章的实数数据，它先计算各值与均值之差的平方和，再除以 N - ddof；N 是参与计算的元素个数。std() 是同一口径方差的平方根。

默认 ddof=0，分母为 N。ddof=1 将分母改为 N - 1；它常用于用随机样本估计总体方差。参数必须与任务的统计口径一致，不是为了获得“更大”或“更小”的结果。

下面 [2, 4, 6] 的均值为 4，偏差平方和为 4 + 0 + 4 = 8。

In [7]:
values = np.array([2.0, 4.0, 6.0])

print(np.var(values, ddof=0))  # 8/3，约 2.6666666667。
print(np.var(values, ddof=1))  # 8/(3-1) = 4.0。
print(np.std(values, ddof=1))  # sqrt(4) = 2.0。

readings = np.array([[2.0, 10.0], [4.0, 14.0], [6.0, 18.0]])
print(np.var(readings, axis=0, ddof=1))  # 两列分别为 [4. 16.]。
print(np.std(readings, axis=0, ddof=1))  # 两列分别为 [2. 4.]。

2.6666666666666665
4.0
2.0
[ 4. 16.]
[2. 4.]


## 8 中位数与分位数

### 8.1 中位数

median() 取排序后居中的值；元素个数为偶数时，取中间两个值的平均。本例六个值已经排序，便于手算。

中位数与算术平均值使用不同的计算规则；报告摘要时应写清是哪一种。

In [8]:
values = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 12.0])

print(np.median(values))  # 4.0，中间的 3 和 5 取平均。
print(np.mean(values))  # 5.0，全部值之和 30 除以 6。

4.0
5.0


### 8.2 分位数与百分位数

quantile() 的 q 是分位水平，取值范围为 0～1；percentile() 使用 0～100。相同方法下，q=0.25 与 25% 对应同一位置。

默认 method="linear"：先排序，用 q × (N - 1) 确定位置，落在相邻位置之间时线性插值；N 是该组元素个数。下面四个值的 25% 位置为 0.25 × 3 = 0.75，对应 0 和 10 之间的 7.5。

多个 q 会在结果最前面增加分位水平轴。method 改变估计规则，结果也可能改变。

In [9]:
values = np.array([0.0, 10.0, 20.0, 30.0])

print(np.quantile(values, [0.25, 0.5, 0.75], method="linear"))
# [7.5 15. 22.5]，分别为三个分位水平的结果。
print(np.percentile(values, [25, 50, 75], method="linear"))  # 相同的三个值。
print(np.quantile(values, 0.25, method="lower"))  # 0.0，改为取较低的相邻值。

table = np.array([[0.0, 10.0, 20.0], [10.0, 30.0, 50.0]])
summary = np.quantile(table, [0.25, 0.75], axis=0, method="linear")
print(summary)  # 两行分别是各列的 25% 与 75% 分位数。
print(summary.shape)  # (2, 3)：两个分位水平、三个原始列。

[ 7.5 15.  22.5]
[ 7.5 15.  22.5]
0.0
[[ 2.5 15.  27.5]
 [ 7.5 25.  42.5]]
(2, 3)


## 9 NaN 感知聚合

本章用浮点 NaN 表示缺失观测。普通 sum()、mean() 等运算会受 NaN 影响；带 nan 前缀的对应函数按各自规则忽略 NaN。它们不会自动判断其他值是否在业务上有效。

nanmean() 使用非 NaN 元素的个数作为分母；nanvar()、nanstd() 的 N 同样是非 NaN 个数。nansum() 将 NaN 当作 0 参与求和。下面每列至少有两个有效观测。

In [10]:
readings = np.array([[2.0, np.nan], [4.0, 10.0], [np.nan, 14.0]])
valid_counts = np.sum(~np.isnan(readings), axis=0)

print(valid_counts)  # [2 2]，每列非 NaN 的个数。
print(np.sum(readings, axis=0))  # [nan nan]。
print(np.nansum(readings, axis=0))  # [6. 24.]。
print(np.nanmean(readings, axis=0))  # [3. 12.]，分母分别为 2。
print(np.nanmax(readings, axis=0))  # [4. 14.]。
print(np.nanvar(readings, axis=0, ddof=1))  # [2. 8.]，分母为 2-1。
print(np.nanstd(readings, axis=0, ddof=1))  # 约 [1.4142 2.8284]。

[2 2]
[nan nan]
[ 6. 24.]
[ 3. 12.]
[ 4. 14.]
[2. 8.]
[1.41421356 2.82842712]


## 10 空输入与无有效观测

### 10.1 空数组与零长度轴

空数组的 sum() 返回加法单位元 0，prod() 返回乘法单位元 1。它们是空聚合的规则，不表示采集到了一个零或一个一。

形状 (0, 3) 表示没有观测行，但仍约定三个字段。沿 axis=0 求和会为三个空组分别返回 0；沿 axis=1 求和则没有任何观测行可输出。

In [11]:
empty = np.array([], dtype=np.float64)
empty_table = np.empty((0, 3), dtype=np.float64)

print(np.sum(empty), np.prod(empty))  # 0.0 1.0。
print(np.sum(empty_table, axis=0))  # [0. 0. 0.]，三个空组。
print(np.sum(empty_table, axis=0).shape)  # (3,)。
print(np.sum(empty_table, axis=1).shape)  # (0,)，没有行结果。
print(empty_table.size)  # 0，没有读取任何未初始化数值元素。

0.0 1.0
[0. 0. 0.]
(3,)
(0,)
0


### 10.2 空最值与 initial

max() 没有默认的空组最大值，不指定 initial 时，空聚合会触发 ValueError。指定 initial 后，它也参与所有非空组的比较，不只是空输入的备用结果。

下面用负无穷 -np.inf 作为浮点最大值计算的初始候选。空组返回负无穷仍是计算约定，应与“实际最大观测值”区分。

In [12]:
empty = np.array([], dtype=np.float64)

# 预期 ValueError：空数组没有可供选取的最大值，此调用也未指定 initial。
np.max(empty)

ValueError: zero-size array to reduction operation maximum which has no identity

In [13]:
print(np.max(empty, initial=-np.inf))  # -inf，来自初始候选。
print(np.max(np.empty((0, 3)), axis=0, initial=-np.inf))  # 三个 -inf。
print(np.max([-5.0, -2.0], initial=0.0))  # 0.0，initial 也参与非空组比较。
print(np.max([-5.0, -2.0], initial=-np.inf))  # -2.0。

-inf
[-inf -inf -inf]
0.0
-2.0


### 10.3 记录空均值的警告

空组没有可用于均值分母的观测数。本例 mean() 返回 NaN 并发出 RuntimeWarning。

warnings.catch_warnings(record=True) 局部收集警告；simplefilter("always") 确保重复运行也能观察。下面打印类型与消息，并用 assert 核对确实产生了 RuntimeWarning；未出现或类型不同会停止执行。退出 with 后恢复原警告设置。

In [14]:
empty_table = np.empty((0, 3), dtype=np.float64)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = np.mean(empty_table, axis=0)

print(result)  # [nan nan nan]，三个字段都没有观测。
for warning in caught:
    print(warning.category.__name__, str(warning.message))
# 本例捕获 RuntimeWarning，消息包含空均值及无效除法。
assert caught and all(warning.category is RuntimeWarning for warning in caught)

[nan nan nan]
RuntimeWarning Mean of empty slice
RuntimeWarning invalid value encountered in divide


### 10.4 全缺失与样本数不足

全 NaN 组的 nansum() 返回 0；nanmean()、nanmax() 返回 NaN 并发出 RuntimeWarning。必须同时看有效数量，不能仅凭总和为 0 认定该组观测总量为零。

nanvar() 与 nanstd() 还要求 N - ddof 大于 0。下面第二列只有一个非 NaN 值，ddof=1 时也无法给出该口径的方差。

In [15]:
readings = np.array([[np.nan, 5.0], [np.nan, np.nan]])

print(np.sum(~np.isnan(readings), axis=0))  # [0 1]，两列有效数量不同。
print(np.nansum(readings, axis=0))  # [0. 5.]，第一列的 0 来自空聚合规则。

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    means = np.nanmean(readings, axis=0)
    maxima = np.nanmax(readings, axis=0)
    variances = np.nanvar(readings, axis=0, ddof=1)

print(means, maxima)  # 均为 [nan 5.]，第一列没有有效数据。
print(variances)  # [nan nan]，两列都不满足 N > 1。
for warning in caught:
    print(warning.category.__name__, str(warning.message))
# 三条 RuntimeWarning 分别说明空均值、全 NaN 最大值及自由度不足。
assert len(caught) == 3 and all(warning.category is RuntimeWarning for warning in caught)

[0 1]
[0. 5.]
[nan  5.] [nan  5.]
[nan nan]
RuntimeWarning Mean of empty slice
RuntimeWarning All-NaN slice encountered
RuntimeWarning Degrees of freedom <= 0 for slice.


## 11 综合应用：两列测量摘要

行表示三次观测，列表示两个测量通道。按忽略 NaN 的口径报告有效数量、均值、最大值和 ddof=1 的标准差；本例两列均有足够观测。

有效数量与摘要一起保留，才能辨认统计分母。

In [16]:
readings = np.array([[2.0, 10.0], [4.0, np.nan], [6.0, 14.0]])
valid_counts = np.sum(~np.isnan(readings), axis=0)
means = np.nanmean(readings, axis=0)
maxima = np.nanmax(readings, axis=0)
deviations = np.nanstd(readings, axis=0, ddof=1)

print("有效数量：", valid_counts)  # [3 2]。
print("均值：", means)  # [4. 12.]。
print("最大值：", maxima)  # [6. 14.]。
print("标准差：", deviations)  # [2. 约2.8284]，分母分别为 3-1、2-1。
print(means.shape, means.dtype)  # (2,) float64，每个通道一个结果。

有效数量： [3 2]
均值： [ 4. 12.]
最大值： [ 6. 14.]
标准差： [2.         2.82842712]
(2,) float64


## 12 选学：同时汇总多个轴

axis 可以是轴编号元组。下面形状 (2, 3, 4) 的三个轴依次表示批次、观测、通道；同时汇总批次和观测轴，保留每个通道的总量。

keepdims=True 会同时保留这两个被汇总的轴。

In [17]:
batches = np.array([
    [[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4]],
    [[10, 20, 30, 40], [10, 20, 30, 40], [10, 20, 30, 40]],
], dtype=np.int64)
totals = np.sum(batches, axis=(0, 1))
kept = np.sum(batches, axis=(0, 1), keepdims=True)

print(totals)  # [33 66 99 132]，每个通道共六个观测。
print(totals.shape, kept.shape)  # (4,) (1, 1, 4)。

[ 33  66  99 132]
(4,) (1, 1, 4)


## 13 选学：协方差与相关系数

cov() 返回协方差矩阵，描述各变量共同变化的情况；对角线是各变量的方差。corrcoef() 返回 Pearson 相关系数矩阵，它用对应变量的标准差对协方差作归一化。

默认 rowvar=True 将每行当作变量。本例每行是一组同时观测、每列是变量，因此使用 rowvar=False。两列的观测必须按同一记录配对，不能各自重排后继续沿用原含义。

本例 cov() 显式设置 ddof=1；没有缺失值，两个变量的方差也都不为零。

In [18]:
paired = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]])
covariance = np.cov(paired, rowvar=False, ddof=1)
correlation = np.corrcoef(paired, rowvar=False)

print(covariance)  # [[1. 2.] [2. 4.]]，对角线与两列 ddof=1 的方差对应。
print(correlation)  # 两变量正向成比例，本例四个系数均为 1.0。
print(covariance.shape, covariance.dtype)  # (2, 2) float64。

repaired = paired.copy()
repaired[:, 1] = repaired[::-1, 1].copy()
print(np.corrcoef(repaired, rowvar=False))
# 仅反转第二列后，非对角线变为 -1.0；这是不同的配对数据。

[[1. 2.]
 [2. 4.]]
[[1. 1.]
 [1. 1.]]
(2, 2) float64
[[ 1. -1.]
 [-1.  1.]]


## 14 选学：频数与区间

### 14.1 histogram

histogram() 按区间统计频数，返回计数与边界。本节使用 density=False，得到每个区间的记录数。

边界 [0, 2, 4] 对应 [0, 2) 和 [2, 4]：除最后一箱外都是左闭右开，最后一箱包含右端点。给定边界之外的数据不计入。

In [19]:
values = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0])
frequencies, edges = np.histogram(values, bins=[0.0, 2.0, 4.0], density=False)

print(frequencies)  # [2 3]，0、1 属于首箱；2、3、4 属于末箱。
print(edges)  # [0. 2. 4.]，三个边界定义两个箱。
print(np.sum(frequencies), values.size)  # 5 7，-1 与 5 在范围外。

[2 3]
[0. 2. 4.]
5 7


### 14.2 bincount

输入已经是一维非负整数编号时，bincount() 直接按编号计数：输出索引表示编号，对应值表示出现次数。minlength 给出输出最小长度，可保留未出现的编号。

它不是连续数值的任意区间分箱；负数或浮点输入不满足这里的输入条件。

In [20]:
labels = np.array([0, 2, 2, 1, 2], dtype=np.int64)
frequencies = np.bincount(labels, minlength=4)

print(frequencies)  # [1 1 3 0]，编号 0～3 分别出现 1、1、3、0 次。
print(frequencies.shape)  # (4,)，编号 3 没出现，但按 minlength 保留。

[1 1 3 0]
(4,)


### 14.3 digitize

digitize() 返回每个值的区间编号，不直接返回频数。bins 必须是一维单调边界；这里使用递增边界和默认 right=False。

对 [0, 2, 4]，编号 1 对应 [0, 2)，编号 2 对应 [2, 4)，低于 0 返回 0，大于或等于 4 返回 3。这个右端点规则与 histogram() 的末箱不同；返回的 3 也不能直接作为三元素边界数组的索引。

In [21]:
values = np.array([-1.0, 0.0, 1.0, 2.0, 4.0, 5.0])
edges = np.array([0.0, 2.0, 4.0])

print(np.digitize(values, edges, right=False))  # [0 1 1 2 3 3]。
print(np.digitize(values, edges, right=True))  # [0 0 1 1 2 3]。
# right=True 改为左开右闭；边界值 0、2、4 的归属随之改变。

[0 1 1 2 3 3]
[0 0 1 1 2 3]


## 本章小结

（1）axis 决定分组方向，keepdims 保留长度为 1 的轴；dtype 影响累计类型与精度。

（2）区分总量、累计结果、最值和最值位置；均值、加权均值与中位数使用不同口径。

（3）方差与标准差必须说明 ddof，分位数必须说明分位水平和 method。

（4）NaN 感知函数仍需配合有效数量；空总和为 0，全缺失均值与不足自由度会给出 NaN 和警告。

（5）initial 参与非空最大值计算；选学中的观测配对、区间端点与编号范围也需要明确。

## 练习

（1）下面两行代表两天，三列代表三个站点。分别计算各站点总量、每天均值，并保留每天均值的二维形状，再计算当日偏差。

In [22]:
counts = np.array([[1, 3, 5], [3, 5, 7]], dtype=np.int64)

# 在此选择 axis 与 keepdims，打印结果。
# 检查：站点总量为 [4 8 12]，每天均值为 [[3.] [5.]]。
# 偏差两行均为 [-2. 0. 2.]，shape 为 (2, 3)。

（2）先预测以下结果，再运行核对。分别解释 ddof、method 和 initial 改变了哪一步计算。

In [23]:
values = np.array([0.0, 10.0, 20.0])

# 先记录预测，再手算均值、偏差平方和、分位位置和最大值候选。
print(np.var(values, ddof=0))
print(np.var(values, ddof=1))
print(np.quantile(values, 0.25, method="linear"))
print(np.quantile(values, 0.25, method="lower"))
print(np.max(values, initial=25.0))

66.66666666666667
100.0
5.0
0.0
25.0


（3）开始时每次测量等权，求各通道均值。随后增加两个条件：NaN 表示缺失，需要忽略；每列至少两次有效测量才报告 ddof=1 的标准差。选择合适函数与检查方法，并说明理由，不能把无有效测量列的结果解释成 0。

In [24]:
readings = np.array([[2.0, np.nan, np.nan], [4.0, 10.0, np.nan], [6.0, np.nan, np.nan]])

# 在此计算有效数量，说明普通 mean 与 nanmean 的选择理由。
# 检查：有效数量为 [3 1 0]，均值为 [4. 10. nan]。
# 如调用 nanmean，用 catch_warnings(record=True) 展示第三列产生的 RuntimeWarning。
# 对标准差先选出有效数量至少为 2 的列，再计算，避免对不足数据作无意义估计。
# 检查：只有第一列满足标准差条件，其 ddof=1 标准差为 2.0。

（4）选学：按边界 [0, 2, 4] 统计下面的区间频数，再计算默认 digitize 区间编号。解释数值 4 在两种结果中的归属为何不同。

In [25]:
values = np.array([0.0, 1.0, 2.0, 4.0])
edges = np.array([0.0, 2.0, 4.0])

# 在此分别调用 histogram 与 digitize，打印计数、边界和编号。
# 检查：histogram 的频数为 [2 2]；digitize 编号为 [1 1 2 3]。
# 在注释中解释末箱右端点规则，并说明编号 3 为什么不能直接索引 edges。

### 重点练习提示

对应第（3）题。先独立完成，再按需要查看提示。

（1）先数每列有多少非缺失值，再区分均值能否定义和样本标准差能否报告。

（2）均值忽略 NaN；标准差只对计数至少为 2 的列计算，未满足条件的位置保留 NaN。

### 重点练习参考解析

对应第（3）题。

三个通道的有效数量为 [3, 1, 0]。第一列均值为 (2+4+6)/3=4，第二列为 10；第三列没有有效观测，应为 NaN。普通 mean 会让含 NaN 的列得到 NaN，因此不能完成第二列的有效均值计算。直接对全部列调用 nanmean 时，应观察全缺失列的 RuntimeWarning。

标准差数组先填 NaN，只对第一列计算：偏差平方和为 4+0+4=8，ddof=1 的分母为 3−1=2，标准差为 2。最终为 [2, NaN, NaN]。第二列虽然有均值，却没有足够观测报告本题规定的标准差；不能用 0 代替。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | 聚合：[sum](https://numpy.org/doc/2.5/reference/generated/numpy.sum.html)、[prod](https://numpy.org/doc/2.5/reference/generated/numpy.prod.html) 的 axis、dtype、keepdims、空输入与整数溢出 Notes；[cumsum](https://numpy.org/doc/2.5/reference/generated/numpy.cumsum.html)、[cumprod](https://numpy.org/doc/2.5/reference/generated/numpy.cumprod.html) 的 axis 与输出形状；[min](https://numpy.org/doc/2.5/reference/generated/numpy.min.html)、[max](https://numpy.org/doc/2.5/reference/generated/numpy.max.html) 的 axis、initial 与 Examples；[argmin](https://numpy.org/doc/2.5/reference/generated/numpy.argmin.html)、[argmax](https://numpy.org/doc/2.5/reference/generated/numpy.argmax.html) 的位置及首次出现规则。统计口径：[mean](https://numpy.org/doc/2.5/reference/generated/numpy.mean.html) 的 dtype 与 Notes（含 float16 的 float32 中间计算）；[average](https://numpy.org/doc/2.5/reference/generated/numpy.average.html) 的 weights 公式；[var](https://numpy.org/doc/2.5/reference/generated/numpy.var.html)、[std](https://numpy.org/doc/2.5/reference/generated/numpy.std.html) 的 ddof 与 Notes；[median](https://numpy.org/doc/2.5/reference/generated/numpy.median.html) 的中间值定义；[quantile](https://numpy.org/doc/2.5/reference/generated/numpy.quantile.html)、[percentile](https://numpy.org/doc/2.5/reference/generated/numpy.percentile.html) 的 q、method、输出形状与线性插值。缺失值：[isnan](https://numpy.org/doc/2.5/reference/generated/numpy.isnan.html)、[nansum](https://numpy.org/doc/2.5/reference/generated/numpy.nansum.html)、[nanmean](https://numpy.org/doc/2.5/reference/generated/numpy.nanmean.html)、[nanmax](https://numpy.org/doc/2.5/reference/generated/numpy.nanmax.html)、[nanvar](https://numpy.org/doc/2.5/reference/generated/numpy.nanvar.html)、[nanstd](https://numpy.org/doc/2.5/reference/generated/numpy.nanstd.html) 的全 NaN、有效计数、ddof 与警告条件。选学：[cov](https://numpy.org/doc/2.5/reference/generated/numpy.cov.html)、[corrcoef](https://numpy.org/doc/2.5/reference/generated/numpy.corrcoef.html) 的定义、rowvar、ddof 与配对观测；[histogram](https://numpy.org/doc/2.5/reference/generated/numpy.histogram.html) 的 density、bins 与 Notes；[bincount](https://numpy.org/doc/2.5/reference/generated/numpy.bincount.html) 的非负整数输入与 minlength；[digitize](https://numpy.org/doc/2.5/reference/generated/numpy.digitize.html) 的 right、单调边界与越界编号。 |
| NumPy 官方源码（GitHub，v2.5.0） | [_core/_methods.py](https://github.com/numpy/numpy/blob/v2.5.0/numpy/_core/_methods.py) 的 _mean：计数为零时发出 RuntimeWarning，再按计数进行除法。本章空均值输出另在 NumPy 2.5.3 实际运行核对。 |
| Python 官方文档（Python 3.12） | [warnings — Testing Warnings](https://docs.python.org/3.12/library/warnings.html#testing-warnings)：catch_warnings(record=True)、simplefilter("always")、警告类别检查和退出后恢复设置。 |